# RO-059 Feedback basics: every number in the Note

The speed loop lives in `images/speedkit.py`: the robot (8 kg, drag 2 N per m/s, motors at most 4 N), the ramp
(1 cm rise per metre, from x = 2 m), the open-loop force and the closed-loop force, and a 100 Hz simulator.
This notebook runs each case and prints the numbers the Note quotes. The noisy run uses a fixed seed.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import speedkit as s
print('ramp pull d = 8 x 9.81 x 0.01 =', round(s.D_RAMP, 4), 'N')
print('largest acceleration from rest = 4 / 8 =', s.U_MAX / s.M, 'm/s^2')

ramp pull d = 8 x 9.81 x 0.01 = 0.7848 N
largest acceleration from rest = 4 / 8 = 0.5 m/s^2


## 1. Open loop: the calibration and where it fails

In [2]:
print('test push 0.5 N settles at', 0.5 / s.B, 'm/s, so 1 N gives', 1 / s.B, 'm/s')
print('open-loop force for 0.5 m/s:', s.u_open(0.5), 'N')
print('steady speed on the ramp :', round(s.steady_open(d=s.D_RAMP), 4), 'm/s')
print('steady speed on carpet   :', round(s.steady_open(b=s.B_CARPET), 4), 'm/s')
ol = s.run(30, closed=False)
print('open loop on the ramp, t = 30 s:', round(ol['v'][-1], 4), 'm/s; x =', round(ol['x'][-1], 2), 'm')
print('reaches the ramp at t =', ol['t'][np.argmax(ol['d'] > 0)], 's')

test push 0.5 N settles at 0.25 m/s, so 1 N gives 0.5 m/s
open-loop force for 0.5 m/s: 1.0 N
steady speed on the ramp : 0.1076 m/s
steady speed on carpet   : 0.3333 m/s
open loop on the ramp, t = 30 s: 0.1082 m/s; x = 6.36 m
reaches the ramp at t = 4.0 s


## 2. Closed loop on the ramp

In [3]:
cl = s.run(30, closed=True)
print('closed loop, t = 30 s:', round(cl['v'][-1], 4), 'm/s; error', round(cl['e'][-1], 4), '; force', round(cl['u'][-1], 4), 'N; x =', round(cl['x'][-1], 2))
print('formula: (1 + 20 x 0.5 - 0.7848) / (2 + 20) =', round(s.steady_closed(d=s.D_RAMP), 4))
print('slow-down open loop  :', round(0.5 - s.steady_open(d=s.D_RAMP), 4), '= d / b')
print('slow-down closed loop:', round(0.5 - s.steady_closed(d=s.D_RAMP), 4), '= d / (b + Kp)')
print('ratio:', round((0.5 - s.steady_open(d=s.D_RAMP)) / (0.5 - s.steady_closed(d=s.D_RAMP)), 3), '= (b + Kp) / b =', (s.B + s.KP) / s.B)
print('largest force in the run:', round(cl['u'].max(), 4), 'N')
for tt in (4.0, 4.01, 4.02, 4.5, 8.0):
    k = int(round(tt / s.DT))
    net = cl['u'][k] - cl['b'][k] * cl['v'][k] - cl['d'][k]
    print(f"t = {tt:5.2f} s: v = {cl['v'][k]:.5f}, e = {cl['e'][k]:.5f}, u = {cl['u'][k]:.4f} N, drag = {cl['b'][k]*cl['v'][k]:.4f} N, net = {net:.4f} N, dv in 10 ms = {net * s.DT / s.M:.6f}")

closed loop, t = 30 s: 0.4643 m/s; error 0.0357 ; force 1.7135 N; x = 14.09
formula: (1 + 20 x 0.5 - 0.7848) / (2 + 20) = 0.4643
slow-down open loop  : 0.3924 = d / b
slow-down closed loop: 0.0357 = d / (b + Kp)
ratio: 11.0 = (b + Kp) / b = 11.0
largest force in the run: 1.7135 N
t =  4.00 s: v = 0.50000, e = 0.00000, u = 1.0000 N, drag = 1.0000 N, net = -0.7848 N, dv in 10 ms = -0.000981
t =  4.01 s: v = 0.49902, e = 0.00098, u = 1.0196 N, drag = 0.9980 N, net = -0.7632 N, dv in 10 ms = -0.000954
t =  4.02 s: v = 0.49806, e = 0.00194, u = 1.0387 N, drag = 0.9961 N, net = -0.7422 N, dv in 10 ms = -0.000928
t =  4.50 s: v = 0.47317, e = 0.02683, u = 1.5365 N, drag = 0.9463 N, net = -0.1946 N, dv in 10 ms = -0.000243
t =  8.00 s: v = 0.46433, e = 0.03567, u = 1.7134 N, drag = 0.9287 N, net = -0.0000 N, dv in 10 ms = -0.000000


### One cycle and the cycles (Figures 5 and 6)

Figure 5's values are the run's first cycle on the ramp. Figure 6 prints each value through `images/loop_cycles.py`'s
`cycle()`, rounded to 4 decimals; needed = drag + ramp and shortfall = needed - push are worked out from the rounded parts.

In [4]:
import loop_cycles as lc
net = cl['u'][400] - s.B * cl['v'][400] - s.D_RAMP
print(f"Figure 5, t = 4.00 s: y_m {cl['y_m'][400]:.3f} m/s, e {cl['e'][400]:.0f} m/s, u {cl['u'][400]:.0f} N, "
      f"drag {s.B * cl['v'][400]:.0f} N, ramp {s.D_RAMP:.3f} N, net force {net:.3f} N, "
      f"speed change {net * s.DT / s.M:.5f} m/s, new speed {cl['v'][401]:.5f} m/s")
print(f"next cycle, t = 4.01 s: e {cl['e'][401]:.5f} m/s, u {cl['u'][401]:.4f} N, "
      f"push change {cl['u'][401] - cl['u'][400]:.2f} N")
print(f"Figure 6: one frame every {lc.STEP} cycles ({lc.STEP * s.DT:.2f} s), {len(lc.TIMES)} frames")
print('     t     y_m       e       u    drag    ramp  needed shortfall  new speed')
for t in lc.TIMES:
    c = lc.cycle(t)
    print(f"{t:6.2f} {c['ym']:.4f} {c['e']:.4f} {c['u']:.4f} {c['drag']:.4f} {c['ramp']:.4f} {c['need']:.4f}    "
          f"{c['short']:.4f}     {c['vnext']:.4f}")

Figure 5, t = 4.00 s: y_m 0.500 m/s, e 0 m/s, u 1 N, drag 1 N, ramp 0.785 N, net force -0.785 N, speed change -0.00098 m/s, new speed 0.49902 m/s
next cycle, t = 4.01 s: e 0.00098 m/s, u 1.0196 N, push change 0.02 N
Figure 6: one frame every 5 cycles (0.05 s), 42 frames
     t     y_m       e       u    drag    ramp  needed shortfall  new speed
  4.00 0.5000 0.0000 1.0000 1.0000 0.7848 1.7848    0.7848     0.4990
  4.05 0.4954 0.0046 1.0929 0.9907 0.7848 1.7755    0.6826     0.4945
  4.10 0.4913 0.0087 1.1736 0.9826 0.7848 1.7674    0.5938     0.4906
  4.15 0.4878 0.0122 1.2439 0.9756 0.7848 1.7604    0.5165     0.4872
  4.20 0.4848 0.0152 1.3050 0.9695 0.7848 1.7543    0.4493     0.4842
  4.25 0.4821 0.0179 1.3581 0.9642 0.7848 1.7490    0.3909     0.4816
  4.30 0.4798 0.0202 1.4044 0.9596 0.7848 1.7444    0.3400     0.4794
  4.35 0.4778 0.0222 1.4446 0.9555 0.7848 1.7403    0.2957     0.4774
  4.40 0.4760 0.0240 1.4796 0.9520 0.7848 1.7368    0.2572     0.4757
  4.45 0.4745 0.0255 1.

## 3. A wrong model: carpet from x = 2 m

In [5]:
olc = s.run(40, closed=False, ramp=False, b_after=s.B_CARPET)
clc = s.run(40, closed=True, ramp=False, b_after=s.B_CARPET)
print('carpet, open loop  :', round(olc['v'][-1], 4), ' formula', round(s.steady_open(b=3), 4))
print('carpet, closed loop:', round(clc['v'][-1], 4), ' formula', round(s.steady_closed(b=3), 4))
for b in (1, 2, 3, 4):
    print('drag', b, ': open', round(s.steady_open(b=b), 3), ' closed', round(s.steady_closed(b=b), 3))
for d in (-1.57, -0.785, 0, 0.785, 1.57):
    print('pull', d, ': open', round(s.steady_open(d=d), 3), ' closed', round(s.steady_closed(d=d), 3))

carpet, open loop  : 0.3333  formula 0.3333
carpet, closed loop: 0.4783  formula 0.4783
drag 1 : open 1.0  closed 0.524
drag 2 : open 0.5  closed 0.5
drag 3 : open 0.333  closed 0.478
drag 4 : open 0.25  closed 0.458
pull -1.57 : open 1.285  closed 0.571
pull -0.785 : open 0.893  closed 0.536
pull 0 : open 0.5  closed 0.5
pull 0.785 : open 0.107  closed 0.464
pull 1.57 : open -0.285  closed 0.429


## 4. The price: sensor noise reaches the motors

In [6]:
cln = s.run(30, closed=True, noise=s.SIGMA_N, seed=1)
k = int(round(10 / s.DT))
print('after 10 s: force std', round(cln['u'][k:].std(), 3), 'N (Kp x sigma =', s.KP * s.SIGMA_N, ')')
print('            true speed std', round(cln['v'][k:].std(), 4), 'm/s; mean', round(cln['v'][k:].mean(), 4))
print('            reading std', round((cln['y_m'][k:] - cln['v'][k:]).std(), 4), 'm/s')

after 10 s: force std 0.402 N (Kp x sigma = 0.4 )
            true speed std 0.0021 m/s; mean 0.464
            reading std 0.02 m/s


## 5. Regulation and tracking

In [7]:
tr = s.run(5, closed=True, ramp=False, v0=0.0, ref=s.ramp_ref)
for tt in (0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 5.0):
    k = int(round(tt / s.DT))
    print(f"t = {tt}: ref {tr['r'][k]:.3f}, v {tr['v'][k]:.4f}, error {tr['e'][k]:.4f}, force {tr['u'][k]:.3f}")
print('predicted lag m a / (b + Kp) =', round(s.M * 0.25 / (s.B + s.KP), 4))
x_ref = np.r_[0, np.cumsum(tr['r'][:-1]) * s.DT]
print('plan ahead of the robot at the end:', round(x_ref[-1] - tr['x'][-1], 4), 'm; robot at', round(tr['x'][-1], 3), 'plan at', round(x_ref[-1], 3))
print('largest force', round(tr['u'].max(), 3))

t = 0.5: ref 0.125, v 0.0566, error 0.0684, force 1.617
t = 1.0: ref 0.250, v 0.1647, error 0.0853, force 2.206
t = 1.5: ref 0.375, v 0.2855, error 0.0895, force 2.540
t = 2.0: ref 0.500, v 0.4094, error 0.0906, force 2.811
t = 2.5: ref 0.500, v 0.4775, error 0.0225, force 1.449
t = 3.0: ref 0.500, v 0.4944, error 0.0056, force 1.111
t = 5.0: ref 0.500, v 0.5000, error 0.0000, force 1.000
predicted lag m a / (b + Kp) = 0.0909
plan ahead of the robot at the end: 0.1818 m; robot at 1.816 plan at 1.998
largest force 2.811
